# Raport comparativ algoritmi TSP

Notebook șablon pentru raportul final cerut în proiect. Acesta:
1. Încarcă seturile de date direct de pe GitHub (raw URL).
2. Rulează toți cei 6 algoritmi pe fiecare set.
3. Generează tabele comparative și grafice (matplotlib + seaborn).

Pentru a rula în **Google Colab**: încarcă acest notebook și înlocuiește URL-ul `GITHUB_BASE` cu repo-ul propriu.

## Setup în Colab

In [ ]:
# Dezarhivează repo-ul în Colab (decomentează la rulare în Colab)
# !git clone https://github.com/<user>/ProiectIA.git
# %cd ProiectIA
# !pip install -q numpy matplotlib requests pandas seaborn

# Pentru rulare locală:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'tsp-app'))

## Importuri și seturi de date

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.algorithms import ALGORITHMS, AlgorithmConfig
from src.io import load_cities_from_csv, load_cities_from_url

GITHUB_BASE = 'https://raw.githubusercontent.com/<user>/ProiectIA/main/tsp-app/data'
DATASETS = {
    'simple_5': f'{GITHUB_BASE}/simple_5.csv',
    'romania_10': f'{GITHUB_BASE}/romania_10.csv',
    'random_25': f'{GITHUB_BASE}/random_25.json',
}

problems = {name: load_cities_from_url(url) for name, url in DATASETS.items()}
for name, p in problems.items():
    print(name, '→', p)

## Configurarea rulărilor

In [ ]:
REPEATS = 3  # pentru algoritmi stocastici — medie pe rulări multiple

CONFIGS = {
    'Backtracking': AlgorithmConfig(max_iterations=10_000_000, time_limit_seconds=30.0),
    'Hill Climbing': AlgorithmConfig(max_iterations=2000, extra={'restarts': 5}),
    'Simulated Annealing': AlgorithmConfig(max_iterations=20_000, extra={'cooling_rate': 0.997}),
    'Genetic Algorithm': AlgorithmConfig(max_iterations=300, extra={'population_size': 100, 'generations': 300}),
    'Ant Colony Optimization': AlgorithmConfig(max_iterations=200, extra={'iterations': 200}),
    'Nearest Neighbor': AlgorithmConfig(),
}

## Rulare comparativă

In [ ]:
records = []
for ds_name, problem in problems.items():
    for algo_name, AlgoClass in ALGORITHMS.items():
        repeats = REPEATS if algo_name in {'Simulated Annealing', 'Genetic Algorithm', 'Ant Colony Optimization', 'Hill Climbing'} else 1
        for run in range(repeats):
            cfg = CONFIGS[algo_name]
            if problem.n > 12 and algo_name == 'Backtracking':
                continue  # impracticabil
            try:
                cfg.seed = run
                result = AlgoClass(cfg).solve(problem)
                records.append({
                    'dataset': ds_name,
                    'algorithm': algo_name,
                    'run': run,
                    'best_length': result.best_length,
                    'elapsed_seconds': result.elapsed_seconds,
                    'iterations': result.iterations,
                })
            except Exception as e:
                print(f'⚠ {algo_name} on {ds_name}: {e}')

df = pd.DataFrame(records)
df

## Statistici agregate

In [ ]:
summary = df.groupby(['dataset', 'algorithm']).agg(
    best_length_mean=('best_length', 'mean'),
    best_length_min=('best_length', 'min'),
    elapsed_mean=('elapsed_seconds', 'mean'),
).round(3)
summary

## Vizualizări

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df, x='algorithm', y='best_length', hue='dataset', ax=axes[0])
axes[0].set_title('Lungime tur (mai mic = mai bun)')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(data=df, x='algorithm', y='elapsed_seconds', hue='dataset', ax=axes[1])
axes[1].set_title('Timp execuție (s)')
axes[1].set_yscale('log')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()